# LeetCode 340 — Longest Substring with At Most K Distinct Characters bằng Python

> **Lưu ý: đây là bài Premium** (bài bị khóa trên LeetCode, chỉ tài khoản trả phí mới mở được).
> Danh mục học của bạn xếp bài này vào nhóm **Hard**; theo những gì mình biết, LeetCode hiện gắn nhãn độ khó Medium (trước đây từng là Hard). Dù nhãn là gì, đây là bài **tổng quát hóa** của LeetCode 159 (k = 2) và là "bài mẹ" của cả một họ bài **sliding window** (cửa sổ trượt) có kích thước thay đổi.

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def lengthOfLongestSubstringKDistinct(self, s: str, k: int) -> int:

```

## Sau notebook này, bạn sẽ nắm được

1. Đề bài yêu cầu gì: **substring** (chuỗi con liên tục) khác **subsequence** ra sao, "at most k distinct" nghĩa là gì, và các trường hợp đặc biệt như **`k = 0`**.
2. Công cụ Python cần dùng: slicing `s[i:j]`, `set`, `dict` đếm tần suất (`get`, `del`), `Counter` và cái bẫy của nó, `min(d, key=d.get)`, và **`OrderedDict`** (`move_to_end`, `popitem(last=False)`).
3. Năm cách giải, đi từ trực giác đến khéo léo:
   - **Cách 1:** Brute force thử **mọi** chuỗi con + `set` — `O(n³)`.
   - **Cách 2:** Cố định đầu trái, mở rộng sang phải, **dừng sớm** khi vượt `k` loại — `O(n²)`.
   - **Cách 3 (cách chính):** **Sliding window** với `dict` **đếm tần suất**, co cửa sổ bằng `while` — `O(n)`.
   - **Cách 4:** Sliding window với **vị trí xuất hiện cuối** của mỗi ký tự; `left` **nhảy một phát** tới `min(last) + 1`. Bản `dict + min` tốn `O(n·k)`, bản **`OrderedDict`** tốn `O(n)` và hợp với câu hỏi phụ (follow-up) "**nếu input là một stream** thì sao?".
   - **Cách 5 (mẹo):** Cửa sổ **không bao giờ co** (non-shrinking window) — chỉ dùng `if`, dịch `left` đúng 1 bước; và **vì sao** mẹo này vẫn đúng.
4. Vì sao mỗi cách đúng (invariant — "bất biến", điều luôn đúng sau mỗi bước), độ phức tạp, lỗi hay gặp.
5. Liên hệ với các bài cùng họ: 159 (k = 2), 904, 3, 992, 1004...

## 1. Đề bài (theo LeetCode)

Given a string `s` and an integer `k`, return *the length of the longest* ***substring*** *of* `s` *that contains at most* `k` ***distinct*** *characters*.

**Example 1**

```text
Input:  s = "eceba", k = 2
Output: 3
Explanation: The substring is "ece" with length 3.
```

**Example 2**

```text
Input:  s = "aa", k = 1
Output: 2
Explanation: The substring is "aa" with length 2.
```

**Constraints**

- `1 <= s.length <= 5 * 10^4`
- `0 <= k <= 50`

### Dịch nhanh

- Cho chuỗi `s` và số nguyên `k`.
- Tìm **chuỗi con liên tục** dài nhất mà bên trong nó có **tối đa `k` loại ký tự khác nhau** (ít hơn `k` loại cũng được).
- Chỉ trả về **độ dài** (một số nguyên), không cần trả về chính chuỗi con.
- Đề **không** giới hạn tập ký tự (không nói "chỉ chữ thường"). Ta coi `s` có thể chứa ký tự bất kỳ; mọi lời giải trong notebook đều dùng `dict`/`set` nên chạy được với mọi ký tự. Chữ hoa và chữ thường là **khác nhau** (`'a'` ≠ `'A'`).
- **`k` có thể bằng 0!** Khi đó không chuỗi con khác rỗng nào hợp lệ (chuỗi 1 ký tự đã có 1 loại > 0), nên đáp án là `0`.

## 2. Hiểu đề bằng tay

### 2.1. Substring khác subsequence

- **Substring** (chuỗi con) = một đoạn **liền nhau** của `s`. Với `s = "eceba"`: `"ece"`, `"ceb"`, `"ba"` là substring.
- **Subsequence** (dãy con) = chọn ký tự theo thứ tự nhưng **được bỏ cách quãng**. `"eea"` là subsequence của `"eceba"` nhưng **không** phải substring.

Đề hỏi **substring**, nên đáp án luôn là một đoạn `s[i..j]` liền mạch.

### 2.2. Đếm "số loại ký tự khác nhau" (distinct) và so với `k`

Với `s = "eceba"`:

| Chuỗi con | Các loại ký tự | Số loại | Hợp lệ khi `k = 1`? | `k = 2`? | `k = 3`? |
|---|---|---|---|---|---|
| `"e"` | {e} | 1 | Có | Có | Có |
| `"ece"` | {e, c} | 2 | Không | Có | Có |
| `"eceb"` | {e, c, b} | 3 | Không | Không | Có |
| `"eceba"` | {e, c, b, a} | 4 | Không | Không | Không |

Ô code dưới in vài chuỗi con của `"eceba"` cùng số loại ký tự, dùng `len(set(...))` (giải thích kỹ ở mục 3).

In [1]:
s = "eceba"
for i, j in [(0, 0), (0, 2), (0, 3), (0, 4), (1, 3), (3, 4)]:
    sub = s[i:j + 1]
    distinct = len(set(sub))
    print(f"s[{i}..{j}] = {sub!r:8} -> {distinct} loại | k=1: {distinct <= 1!s:5} k=2: {distinct <= 2!s:5} k=3: {distinct <= 3}")

s[0..0] = 'e'      -> 1 loại | k=1: True  k=2: True  k=3: True
s[0..2] = 'ece'    -> 2 loại | k=1: False k=2: True  k=3: True
s[0..3] = 'eceb'   -> 3 loại | k=1: False k=2: False k=3: True
s[0..4] = 'eceba'  -> 4 loại | k=1: False k=2: False k=3: False
s[1..3] = 'ceb'    -> 3 loại | k=1: False k=2: False k=3: True
s[3..4] = 'ba'     -> 2 loại | k=1: False k=2: True  k=3: True


### 2.3. Đáp án thay đổi thế nào khi `k` tăng?

Với cùng `s = "eceba"`, thử tính tay:

| `k` | Chuỗi con dài nhất (một ví dụ) | Đáp án |
|---|---|---|
| 0 | (chỉ có chuỗi rỗng) | 0 |
| 1 | `"e"` (không có 2 ký tự giống nhau liền nhau) | 1 |
| 2 | `"ece"` | 3 |
| 3 | `"eceb"` | 4 |
| 4 | `"eceba"` (cả chuỗi) | 5 |
| 5, 6, ... | cả chuỗi | 5 |

Ba **edge case** (trường hợp biên) rút ra:

- `k = 0` ⇒ đáp án `0`.
- `k = 1` ⇒ đáp án là **đoạn dài nhất toàn một ký tự** (ví dụ `"aabbbc"` → `"bbb"` → 3).
- `k ≥` số loại ký tự của cả `s` ⇒ đáp án là **cả chuỗi**, `len(s)`.

Đáp án **không giảm** khi `k` tăng: chuỗi con nào hợp lệ với `k` thì cũng hợp lệ với `k + 1`.

In [2]:
s = "eceba"
print("Số loại ký tự của cả chuỗi:", len(set(s)))   # 4 -> với k >= 4, đáp án = len(s) = 5

s2 = "aabbbc"
# k = 1: tìm đoạn dài nhất toàn một ký tự bằng cách đếm "chuỗi chạy" (run)
best_run, run = 1, 1
for i in range(1, len(s2)):
    run = run + 1 if s2[i] == s2[i - 1] else 1
    best_run = max(best_run, run)
print("Đoạn dài nhất toàn một ký tự trong", repr(s2), "dài:", best_run)   # 3 ('bbb')

Số loại ký tự của cả chuỗi: 4
Đoạn dài nhất toàn một ký tự trong 'aabbbc' dài: 3


## 3. Nền tảng Python

### 3.1. Chỉ số (index) và slicing `s[i:j]`

- `s[i]` là ký tự ở vị trí `i` (đếm từ `0`).
- `s[i:j]` lấy các ký tự từ `i` **đến trước** `j` — không lấy `j`. Độ dài kết quả là `j - i`.
- Trong notebook này ta hay nói "đoạn `[left..right]`" (lấy **cả hai đầu**). Đoạn đó trong Python là `s[left:right + 1]`, và độ dài là **`right - left + 1`** (đừng quên `+ 1`!).
- Nếu `left = right + 1` thì đoạn **rỗng**, độ dài `0` — tình huống này sẽ xuất hiện khi `k = 0`.

### 3.2. `set` — tập hợp, tự bỏ phần tử trùng

`set(chuỗi)` tạo một **tập hợp** các ký tự, mỗi ký tự chỉ giữ **một bản**. Vì vậy `len(set(sub))` = số loại ký tự khác nhau.

Ta cũng có thể tạo `set` rỗng rồi thêm dần bằng `.add(x)`: thêm một ký tự đã có thì `set` không đổi. Cách 2 dùng đúng điều này để đếm số loại khi mở rộng dần sang phải.

Ô code dưới thử cả hai ý: slicing (3.1) rồi `set` (3.2).

In [3]:
s = "aabacbebebe"
print(s[4], s[6], s[10])         # c e e
left, right = 4, 10
print(s[left:right + 1], "dài", right - left + 1)   # 'cbebebe' dài 7
left, right = 3, 2               # left = right + 1 -> đoạn rỗng
print(repr(s[left:right + 1]), "dài", right - left + 1)   # '' dài 0

print(set("aabbb"), len(set("aabbb")))     # {'a', 'b'} 2   (thứ tự in của set có thể khác)
seen = set()
for ch in "abacd":
    seen.add(ch)
    print(f"thêm {ch!r} -> có {len(seen)} loại")

c e e
cbebebe dài 7
'' dài 0
{'b', 'a'} 2
thêm 'a' -> có 1 loại
thêm 'b' -> có 2 loại
thêm 'a' -> có 2 loại
thêm 'c' -> có 3 loại
thêm 'd' -> có 4 loại


### 3.3. `dict` đếm tần suất — trái tim của sliding window

`set` chỉ biết "có / không". Khi cửa sổ **bỏ bớt** một ký tự ở đầu trái, ta cần biết ký tự đó **còn bản nào khác** trong cửa sổ không. Vì thế ta **đếm** bằng `dict`:

- `count.get(ch, 0)` đọc số lần của `ch`; nếu chưa có key thì trả `0` (không báo lỗi như `count[ch]`).
- Thêm: `count[ch] = count.get(ch, 0) + 1`.
- Bớt: `count[ch] -= 1`; nếu về `0` thì **`del count[ch]`** để xóa hẳn key.
- Nhờ luôn xóa key bằng 0, ta có quy tắc vàng: **`len(count)` = số loại ký tự đang có trong cửa sổ.**

### 3.4. `Counter` — tiện nhưng có bẫy

`collections.Counter` là một `dict` chuyên đếm: `Counter("abca")` → `{'a': 2, 'b': 1, 'c': 1}`, và đọc key chưa có thì trả `0`.

**Bẫy:** khi `c[x] -= 1` về `0`, `Counter` **vẫn giữ key** với giá trị 0. Nên `len(c)` **không còn** bằng số loại ký tự nữa. Phải tự `del` (hoặc đếm số key có value > 0).

Ô code dưới làm cả hai: đếm bằng `dict` (thêm, bớt, `del`), rồi xem bẫy của `Counter`.

In [4]:
count = {}
for ch in "abca":                      # cửa sổ "abca"
    count[ch] = count.get(ch, 0) + 1
print(count, "-> số loại:", len(count))    # {'a': 2, 'b': 1, 'c': 1} -> 3

for out in "ab":                       # bỏ 'a' rồi bỏ 'b' ở đầu trái -> cửa sổ "ca"
    count[out] -= 1
    if count[out] == 0:
        del count[out]
    print(f"bỏ {out!r} -> {count}, số loại: {len(count)}")

# --- bẫy của Counter ---
from collections import Counter
c = Counter("abca")
print(c, len(c))            # 3 loại
c["b"] -= 1                 # 'b' về 0 nhưng KHÔNG bị xóa
print(c, "len =", len(c))   # vẫn 3  -> sai nếu dùng len làm "số loại"
del c["b"]
print(c, "len =", len(c))   # 2  -> đúng

{'a': 2, 'b': 1, 'c': 1} -> số loại: 3
bỏ 'a' -> {'a': 1, 'b': 1, 'c': 1}, số loại: 3
bỏ 'b' -> {'a': 1, 'c': 1}, số loại: 2
Counter({'a': 2, 'b': 1, 'c': 1}) 3
Counter({'a': 2, 'c': 1, 'b': 0}) len = 3
Counter({'a': 2, 'c': 1}) len = 2


### 3.5. `min(d, key=d.get)` — tìm key có value nhỏ nhất

Ở Cách 4, ta lưu `last[ch]` = vị trí xuất hiện **cuối** của `ch`. Khi cần bỏ một loại, ta bỏ loại có `last` **nhỏ nhất**:

- `min(last)` sẽ so sánh **key** (so chữ cái) — **không** phải điều ta muốn.
- `min(last, key=last.get)` duyệt các key nhưng so bằng `last.get(key)` = value ⇒ trả về **key** có value nhỏ nhất.
- Thao tác này phải duyệt hết `len(last)` key ⇒ tốn `O(k)`.

In [5]:
last = {"c": 4, "a": 3, "b": 5, "e": 6}
print(min(last))                   # 'a'  (so key theo bảng chữ cái — chỉ tình cờ trùng)
print(min(last, key=last.get))     # 'a'  (so theo value: 3 là nhỏ nhất)
last2 = {"z": 1, "a": 9}
print(min(last2), "vs", min(last2, key=last2.get))   # 'a' vs 'z' -> thấy rõ khác nhau

a
a
a vs z


### 3.6. `OrderedDict` — `dict` nhớ thứ tự và sắp lại được

`collections.OrderedDict` giữ các key theo **thứ tự**, và có 2 thao tác `O(1)` mà Cách 4 cần:

- `od.move_to_end(key)` — chuyển key về **cuối** hàng.
- `od.popitem(last=False)` — lấy ra (và xóa) cặp `(key, value)` ở **đầu** hàng.

**Điểm dễ nhầm:** gán lại `od[key] = giá_trị_mới` cho key **đã có** chỉ đổi value, **không** đổi vị trí. Muốn key thành "mới nhất" thì phải `move_to_end`.

(`dict` thường từ Python 3.7 cũng nhớ thứ tự chèn, nhưng không có `move_to_end`; muốn đưa key về cuối phải `del` rồi gán lại.)

In [6]:
from collections import OrderedDict
od = OrderedDict()
od["a"] = 0; od["b"] = 1; od["c"] = 2
print(list(od.items()))           # [('a', 0), ('b', 1), ('c', 2)]
od["a"] = 3                       # đổi value, KHÔNG đổi vị trí
print(list(od.items()))           # [('a', 3), ('b', 1), ('c', 2)]
od.move_to_end("a")               # giờ 'a' mới là "mới nhất"
print(list(od.items()))           # [('b', 1), ('c', 2), ('a', 3)]
print(od.popitem(last=False))     # ('b', 1)  <- cặp cũ nhất
print(list(od.items()))           # [('c', 2), ('a', 3)]

[('a', 0), ('b', 1), ('c', 2)]
[('a', 3), ('b', 1), ('c', 2)]
[('b', 1), ('c', 2), ('a', 3)]
('b', 1)
[('c', 2), ('a', 3)]


### 3.7. "Cửa sổ" `[left..right]` và `while` khác `if`

**Sliding window** dùng **hai con trỏ** (two pointers) `left` và `right` đánh dấu hai đầu đoạn đang xét:

- `right` tiến từng bước để **mở rộng** cửa sổ sang phải.
- `left` tiến để **co** cửa sổ từ bên trái khi nó vi phạm điều kiện.
- Cả hai **chỉ đi tới, không bao giờ lùi**.

Để co, có hai lựa chọn rất khác nhau:

- `if điều_kiện: làm 1 lần` — kiểm tra **một lần**.
- `while điều_kiện: làm ...` — lặp **cho tới khi** điều kiện hết đúng.

Cách 3 dùng `while` (co tới khi hợp lệ hẳn), Cách 5 cố tình dùng `if` (chỉ co 1 bước).

In [7]:
s = "aabac"
left = 0
for right in range(len(s)):
    print(f"right={right}: cửa sổ [{left}..{right}] = {s[left:right + 1]!r}")
    if right == 3:
        left = 2          # giả vờ co một phát để thấy left chỉ đi tới
x = 7
if x > 3:
    x -= 2
print("sau if   :", x)    # 5  (chỉ trừ 1 lần)
x = 7
while x > 3:
    x -= 2
print("sau while:", x)    # 3  (trừ tới khi x <= 3)

right=0: cửa sổ [0..0] = 'a'
right=1: cửa sổ [0..1] = 'aa'
right=2: cửa sổ [0..2] = 'aab'
right=3: cửa sổ [0..3] = 'aaba'
right=4: cửa sổ [2..4] = 'bac'
sau if   : 5
sau while: 3


## 4. Cách 1 — Brute force: thử mọi chuỗi con

Cách người mới nghĩ ra đầu tiên: **liệt kê mọi** đoạn `s[i..j]`, đếm số loại bằng `len(set(...))`, đoạn nào `<= k` loại thì cập nhật kỷ lục.

```text
best = 0
với mọi i từ 0..n-1:
    với mọi j từ i..n-1:
        nếu len(set(s[i..j])) <= k: best = max(best, j - i + 1)
```

`k = 0` được xử lý tự nhiên: không đoạn nào có `<= 0` loại, `best` giữ nguyên `0`.

Cách này chậm nhưng **rất đáng tin** — ta sẽ dùng nó làm "trọng tài" để kiểm tra các cách nhanh.

In [8]:
def longest_k_distinct_bruteforce(s: str, k: int) -> int:
    n = len(s)
    best = 0
    for i in range(n):
        for j in range(i, n):
            if len(set(s[i:j + 1])) <= k:      # cắt chuỗi + tạo set: O(độ dài đoạn)
                best = max(best, j - i + 1)
    return best

print(longest_k_distinct_bruteforce("eceba", 2))        # 3
print(longest_k_distinct_bruteforce("aa", 1))           # 2
print(longest_k_distinct_bruteforce("aabacbebebe", 3))  # 7 ('cbebebe')
print(longest_k_distinct_bruteforce("abc", 0))          # 0
print([longest_k_distinct_bruteforce("eceba", k) for k in range(6)])   # [0, 1, 3, 4, 5, 5]

3
2
7
0
[0, 1, 3, 4, 5, 5]


### Cách 1 chậm cỡ nào?

Có khoảng `n(n+1)/2` đoạn, mỗi đoạn tốn tới `n` bước để cắt và dựng `set` ⇒ **Time `O(n³)`**, **Space `O(n)`** (chuỗi con tạm).

Với `n = 5 * 10^4` (giới hạn đề): `n³ ≈ 1.25 * 10^14` thao tác — Python làm khoảng `10^7` thao tác đơn giản/giây ⇒ mất hàng năm. Ô dưới đếm số ký tự được "đụng tới" để thấy tốc độ tăng: `n` gấp đôi ⇒ công việc gấp khoảng **8 lần**.

In [9]:
def count_work_bruteforce(n):
    # tổng độ dài của mọi đoạn s[i..j] = số ký tự phải cắt + đưa vào set
    return sum(j - i + 1 for i in range(n) for j in range(i, n))

for n in [10, 20, 40, 80]:
    print(f"n = {n:3}: {count_work_bruteforce(n):>7,} ký tự được xử lý")

n =  10:     220 ký tự được xử lý
n =  20:   1,540 ký tự được xử lý
n =  40:  11,480 ký tự được xử lý
n =  80:  88,560 ký tự được xử lý


## 5. Cách 2 — Cố định trái, mở rộng phải, dừng sớm: `O(n²)`

### Cách 1 lãng phí ở đâu?

Với cùng `i`, Cách 1 dựng lại `set(s[i:j+1])` **từ đầu** cho mỗi `j`. Nhưng `s[i..j+1]` chỉ hơn `s[i..j]` đúng **một** ký tự `s[j+1]`!

### Hai cải tiến

1. **Dùng chung `set`:** khi `j` tăng, chỉ cần `seen.add(s[j])` — `O(1)` trung bình.
2. **Dừng sớm:** khi `len(seen) > k`, mọi đoạn dài hơn (cùng đầu `i`) cũng có `> k` loại (thêm ký tự không làm **giảm** số loại) ⇒ `break` ngay.

Lưu ý thứ tự: **thêm** `s[j]` → nếu vượt `k` thì **dừng** → nếu không thì **ghi nhận** độ dài `j - i + 1`. Với `k = 0`, ngay ký tự đầu tiên đã vượt nên vòng trong `break` liền, `best` giữ `0`.

In [10]:
def longest_k_distinct_quadratic(s: str, k: int) -> int:
    n = len(s)
    best = 0
    for i in range(n):
        seen = set()
        for j in range(i, n):
            seen.add(s[j])
            if len(seen) > k:          # vượt k loại -> mọi j lớn hơn cũng vượt
                break
            best = max(best, j - i + 1)
    return best

print(longest_k_distinct_quadratic("eceba", 2))        # 3
print(longest_k_distinct_quadratic("aa", 1))           # 2
print(longest_k_distinct_quadratic("aabacbebebe", 3))  # 7
print(longest_k_distinct_quadratic("abc", 0))          # 0

3
2
7
0


### Trace Cách 2 với `s = "eceba"`, `k = 2`

| `i` (trái) | Mở rộng `j` | `seen` khi dừng | Lý do dừng | Đoạn tốt nhất từ `i` |
|---|---|---|---|---|
| 0 | e, c, e, **b** | {e, c, b} | `b` là loại thứ 3 | `"ece"` (3) |
| 1 | c, e, **b** | {c, e, b} | `b` là loại thứ 3 | `"ce"` (2) |
| 2 | e, b, **a** | {e, b, a} | `a` là loại thứ 3 | `"eb"` (2) |
| 3 | b, a | {b, a} | hết chuỗi | `"ba"` (2) |
| 4 | a | {a} | hết chuỗi | `"a"` (1) |

`best = 3`.

In [11]:
def trace_quadratic(s, k):
    best = 0
    for i in range(len(s)):
        seen, j = set(), i
        while j < len(s):
            seen.add(s[j])
            if len(seen) > k:
                break
            j += 1
        reason = f"dừng ở s[{j}]={s[j]!r} (loại thứ {k + 1})" if j < len(s) else "hết chuỗi"
        best = max(best, j - i)
        print(f"i={i}: đoạn hợp lệ dài nhất {s[i:j]!r:7} (dài {j - i}), {reason}, best={best}")
    return best

print("Kết quả:", trace_quadratic("eceba", 2))

i=0: đoạn hợp lệ dài nhất 'ece'   (dài 3), dừng ở s[3]='b' (loại thứ 3), best=3
i=1: đoạn hợp lệ dài nhất 'ce'    (dài 2), dừng ở s[3]='b' (loại thứ 3), best=3
i=2: đoạn hợp lệ dài nhất 'eb'    (dài 2), dừng ở s[4]='a' (loại thứ 3), best=3
i=3: đoạn hợp lệ dài nhất 'ba'    (dài 2), hết chuỗi, best=3
i=4: đoạn hợp lệ dài nhất 'a'     (dài 1), hết chuỗi, best=3
Kết quả: 3


### Điểm yếu của Cách 2

Dừng sớm chỉ giúp khi ký tự "vượt `k`" xuất hiện sớm. Trường hợp xấu nhất: `s` có ít loại ký tự, ví dụ `s = "a" * n`, `k = 1`. Với mỗi `i`, vòng trong chạy tới **hết chuỗi**: tổng `n + (n-1) + ... + 1 ≈ n²/2` bước.

Với `n = 5 * 10^4`: `≈ 1.25 * 10^9` bước ⇒ quá chậm với Python (Time Limit Exceeded). **Time `O(n²)`**, **Space `O(k)`** (`seen` có tối đa `k + 1` phần tử).

Ô dưới đếm số bước vòng trong:

In [12]:
def count_steps_quadratic(s, k):
    steps = 0
    for i in range(len(s)):
        seen = set()
        for j in range(i, len(s)):
            steps += 1
            seen.add(s[j])
            if len(seen) > k:
                break
    return steps

for n in [1000, 2000, 4000]:
    print(f"n = {n}: 'a'*n, k=1 -> {count_steps_quadratic('a' * n, 1):>9,} bước  (n²/2 ≈ {n * n // 2:,})")

n = 1000: 'a'*n, k=1 ->   500,500 bước  (n²/2 ≈ 500,000)


n = 2000: 'a'*n, k=1 -> 2,001,000 bước  (n²/2 ≈ 2,000,000)


n = 4000: 'a'*n, k=1 -> 8,002,000 bước  (n²/2 ≈ 8,000,000)


## 6. Bước đột phá: `left` tốt nhất chỉ đi tới, không bao giờ lùi

Hãy đổi góc nhìn: thay vì cố định đầu **trái**, ta cố định đầu **phải** `right` và hỏi:

> Với mỗi `right`, đầu trái **nhỏ nhất** `L(right)` sao cho `s[L..right]` còn hợp lệ (`<= k` loại) là bao nhiêu?

Đoạn dài nhất kết thúc tại `right` khi đó dài `right - L(right) + 1`, và đáp án là `max` của các độ dài đó.

**Hai quan sát then chốt:**

1. **Co lại vẫn hợp lệ:** nếu `s[l..r]` có `<= k` loại thì mọi đoạn con bên trong cũng `<= k` loại (bớt ký tự không làm tăng số loại).
2. **`L(right)` không giảm:** giả sử `L(r+1) < L(r)`. Thì `s[L(r+1)..r+1]` hợp lệ ⇒ đoạn con `s[L(r+1)..r]` cũng hợp lệ (quan sát 1) ⇒ `L(r)` lẽ ra phải `<= L(r+1)` — mâu thuẫn. Vậy `L(r+1) >= L(r)`.

Hệ quả: khi `right` tăng, `left` chỉ cần **đi tới** từ chỗ cũ, không bao giờ phải quay lại. Hai con trỏ cùng đi một chiều ⇒ tổng số bước ≤ `2n`. Ô dưới kiểm chứng bằng cách tính `L(right)` "thô" (chậm) cho vài chuỗi:

In [13]:
def smallest_valid_left(s, k):
    # với mỗi right: lùi L sang trái chừng nào s[L-1..right] vẫn hợp lệ (cách chậm, chỉ để quan sát)
    result = []
    for right in range(len(s)):
        L = right + 1                      # bắt đầu từ đoạn rỗng
        while L - 1 >= 0 and len(set(s[L - 1:right + 1])) <= k:
            L -= 1
        result.append(L)
    return result

for s, k in [("eceba", 2), ("aabacbebebe", 3), ("abc", 0)]:
    Ls = smallest_valid_left(s, k)
    lengths = [r - L + 1 for r, L in enumerate(Ls)]
    print(f"{s!r:14} k={k}: L = {Ls} | độ dài = {lengths} | max = {max(lengths)}")

'eceba'        k=2: L = [0, 0, 0, 2, 3] | độ dài = [1, 2, 3, 2, 2] | max = 3
'aabacbebebe'  k=3: L = [0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 4] | độ dài = [1, 2, 3, 4, 5, 6, 3, 4, 5, 6, 7] | max = 7
'abc'          k=0: L = [1, 2, 3] | độ dài = [0, 0, 0] | max = 0


### Đọc kết quả

- `"aabacbebebe"`, `k = 3`: `L = [0,0,0,0,0,0,4,4,4,4,4]` — không bao giờ giảm. Tại `right = 6` (chữ `e`, loại thứ 4), `left` phải nhảy từ `0` lên `4` để bỏ **hết** chữ `a`; sau đó đứng yên. Đáp án `max = 7`.
- `"abc"`, `k = 0`: `L(right) = right + 1` — đoạn luôn **rỗng**, độ dài `0`. Đây là cách `k = 0` "tự nhiên" chạy đúng trong sliding window.

Vậy nhiệm vụ của sliding window là: **duy trì `left = L(right)` một cách rẻ tiền** khi `right` tiến dần. Cách 3, 4, 5 khác nhau ở chỗ "duy trì" bằng công cụ gì.

## 7. Cách 3 (cách chính) — Sliding window + `dict` đếm tần suất

Giữ `count` = bảng đếm các ký tự **trong cửa sổ** `s[left..right]`. Nhờ luôn xóa key về 0, `len(count)` = số loại ký tự trong cửa sổ.

Mỗi vòng lặp với `right = 0, 1, ..., n-1`:

1. **Mở rộng:** thêm `ch = s[right]` vào `count`.
2. **Co cho tới khi hợp lệ:** `while len(count) > k`: bớt `s[left]` khỏi `count` (xóa key nếu về 0), `left += 1`.
3. **Ghi nhận:** lúc này cửa sổ hợp lệ và chính là `s[L(right)..right]`, nên `best = max(best, right - left + 1)`.

Thứ tự **thêm → co → ghi nhận** là bắt buộc: ghi nhận trước khi co sẽ tính cả cửa sổ chưa hợp lệ (xem mục lỗi thường gặp).

So với bài 159 (k = 2): chỉ cần thay hằng số `2` bằng biến `k` — đó là lợi ích của việc hiểu **tại sao** thuật toán đúng thay vì nhớ code.

`enumerate(s)` trả về từng cặp `(chỉ số, ký tự)`: `(0, 'e'), (1, 'c'), ...` — tiện vì ta cần cả `right` lẫn `ch`.

**Còn `k = 0`?** Ta **không cần** viết `if k == 0: return 0`. Thêm `s[right]` ⇒ `len(count) = 1 > 0` ⇒ `while` bỏ luôn chính ký tự đó, `left = right + 1` ⇒ cửa sổ rỗng, độ dài `0`. (Nhiều người vẫn thêm dòng `if k == 0` cho dễ đọc — cũng được.) Code và chạy thử:

In [14]:
def longest_k_distinct_window(s: str, k: int) -> int:
    count = {}          # ký tự -> số lần xuất hiện trong s[left..right]
    left = 0
    best = 0
    for right, ch in enumerate(s):
        # 1) mở rộng
        count[ch] = count.get(ch, 0) + 1
        # 2) co cho tới khi còn tối đa k loại
        while len(count) > k:
            out = s[left]
            count[out] -= 1
            if count[out] == 0:
                del count[out]
            left += 1
        # 3) ghi nhận cửa sổ hợp lệ
        best = max(best, right - left + 1)
    return best


tests = [("eceba", 2), ("aa", 1), ("aabacbebebe", 3), ("abc", 0), ("a", 0),
         ("aabbbc", 1), ("abc", 10), ("abcdef", 2)]
for s, k in tests:
    print(f"s={s!r:14} k={k:2} -> {longest_k_distinct_window(s, k)}")

s='eceba'        k= 2 -> 3
s='aa'           k= 1 -> 2
s='aabacbebebe'  k= 3 -> 7
s='abc'          k= 0 -> 0
s='a'            k= 0 -> 0
s='aabbbc'       k= 1 -> 3
s='abc'          k=10 -> 3
s='abcdef'       k= 2 -> 2


## 8. Mô phỏng Cách 3

### 8.1. Trace ví dụ đề `s = "eceba"`, `k = 2`

| `right` | `ch` | `count` sau khi thêm | Co? | `left` | Cửa sổ | `best` |
|---|---|---|---|---|---|---|
| 0 | e | {e:1} | không | 0 | `e` | 1 |
| 1 | c | {e:1, c:1} | không | 0 | `ec` | 2 |
| 2 | e | {e:2, c:1} | không | 0 | `ece` | 3 |
| 3 | b | {e:2, c:1, b:1} → 3 loại | bỏ `e`@0 (e:1), bỏ `c`@1 (xóa c) | 2 | `eb` | 3 |
| 4 | a | {e:1, b:1, a:1} → 3 loại | bỏ `e`@2 (xóa e) | 3 | `ba` | 3 |

Dòng `right = 3`: bỏ `e`@0 **chưa đủ** (vẫn còn `e` ở vị trí 2), phải bỏ tiếp `c`@1 thì `c` mới **biến mất hẳn**. Đó là lý do cần `while`.

In [15]:
def trace_window(s, k):
    count, left, best = {}, 0, 0
    for right, ch in enumerate(s):
        count[ch] = count.get(ch, 0) + 1
        note = f"  ({len(count)} loại > k={k}!)" if len(count) > k else ""
        print(f"right={right} ch={ch!r}: thêm -> {count}{note}")
        removed = []
        while len(count) > k:
            out = s[left]
            count[out] -= 1
            if count[out] == 0:
                del count[out]
            removed.append(f"{out}@{left}")
            left += 1
        if removed:
            print(f"    co: bỏ {', '.join(removed)} -> {count}, left={left}")
        best = max(best, right - left + 1)
        print(f"    cửa sổ [{left}..{right}] = {s[left:right + 1]!r}, dài {right - left + 1}, best={best}")
    return best

print("Kết quả:", trace_window("eceba", 2))

right=0 ch='e': thêm -> {'e': 1}
    cửa sổ [0..0] = 'e', dài 1, best=1
right=1 ch='c': thêm -> {'e': 1, 'c': 1}
    cửa sổ [0..1] = 'ec', dài 2, best=2
right=2 ch='e': thêm -> {'e': 2, 'c': 1}
    cửa sổ [0..2] = 'ece', dài 3, best=3
right=3 ch='b': thêm -> {'e': 2, 'c': 1, 'b': 1}  (3 loại > k=2!)
    co: bỏ e@0, c@1 -> {'e': 1, 'b': 1}, left=2
    cửa sổ [2..3] = 'eb', dài 2, best=3
right=4 ch='a': thêm -> {'e': 1, 'b': 1, 'a': 1}  (3 loại > k=2!)
    co: bỏ e@2 -> {'b': 1, 'a': 1}, left=3
    cửa sổ [3..4] = 'ba', dài 2, best=3
Kết quả: 3


### 8.2. Trace "khó" hơn: `s = "aabacbebebe"`, `k = 3`

Ví dụ này cho thấy `while` có thể phải bỏ **nhiều** ký tự một lúc — kể cả ký tự **không** phải loại bị loại bỏ (ở đây `b`@2), vì cửa sổ phải **liền mạch**:

| `right` | `ch` | `count` sau khi thêm | Co? | `left` | Cửa sổ | `best` |
|---|---|---|---|---|---|---|
| 0–5 | a,a,b,a,c,b | {a:3, b:2, c:1} | không | 0 | `aabacb` | 6 |
| 6 | e | {a:3, b:2, c:1, e:1} → 4 loại | bỏ `a`@0, `a`@1, `b`@2, `a`@3 (xóa a) | 4 | `cbe` | 6 |
| 7 | b | {b:2, c:1, e:1} | không | 4 | `cbeb` | 6 |
| 8 | e | {b:2, c:1, e:2} | không | 4 | `cbebe` | 6 |
| 9 | b | {b:3, c:1, e:2} | không | 4 | `cbebeb` | 6 |
| 10 | e | {b:3, c:1, e:3} | không | 4 | `cbebebe` | **7** |

Tại `right = 6` cửa sổ **ngắn đi** từ 6 xuống 3, `best` vẫn giữ 6; bốn bước sau cửa sổ dài lại và vượt kỷ lục.

In [16]:
print("Kết quả:", trace_window("aabacbebebe", 3))

right=0 ch='a': thêm -> {'a': 1}
    cửa sổ [0..0] = 'a', dài 1, best=1
right=1 ch='a': thêm -> {'a': 2}
    cửa sổ [0..1] = 'aa', dài 2, best=2
right=2 ch='b': thêm -> {'a': 2, 'b': 1}
    cửa sổ [0..2] = 'aab', dài 3, best=3
right=3 ch='a': thêm -> {'a': 3, 'b': 1}
    cửa sổ [0..3] = 'aaba', dài 4, best=4
right=4 ch='c': thêm -> {'a': 3, 'b': 1, 'c': 1}
    cửa sổ [0..4] = 'aabac', dài 5, best=5
right=5 ch='b': thêm -> {'a': 3, 'b': 2, 'c': 1}
    cửa sổ [0..5] = 'aabacb', dài 6, best=6
right=6 ch='e': thêm -> {'a': 3, 'b': 2, 'c': 1, 'e': 1}  (4 loại > k=3!)
    co: bỏ a@0, a@1, b@2, a@3 -> {'b': 1, 'c': 1, 'e': 1}, left=4
    cửa sổ [4..6] = 'cbe', dài 3, best=6
right=7 ch='b': thêm -> {'b': 2, 'c': 1, 'e': 1}
    cửa sổ [4..7] = 'cbeb', dài 4, best=6
right=8 ch='e': thêm -> {'b': 2, 'c': 1, 'e': 2}
    cửa sổ [4..8] = 'cbebe', dài 5, best=6
right=9 ch='b': thêm -> {'b': 3, 'c': 1, 'e': 2}
    cửa sổ [4..9] = 'cbebeb', dài 6, best=6
right=10 ch='e': thêm -> {'b': 3, 'c': 1, 'e': 3

### 8.3. Vì sao Cách 3 là `O(n)` dù có `while` lồng trong `for`?

"Vòng `while` nằm trong vòng `for`" rất dễ khiến ta tưởng là `O(n²)`. Hãy đếm theo **từng vị trí** thay vì theo từng vòng:

- Mỗi vị trí được **thêm** vào cửa sổ đúng **1 lần** (khi `right` đi qua nó).
- Mỗi vị trí bị **bỏ** khỏi cửa sổ **nhiều nhất 1 lần** (khi `left` đi qua nó), vì `left` chỉ tăng và không vượt `n`.

Tổng số lần chạy thân `while` trong **cả** thuật toán ≤ `n`. Cộng lại ≤ `2n` thao tác ⇒ **`O(n)`**, không phụ thuộc `k`. Kiểu lập luận "tính tổng cả quá trình" gọi là **phân tích khấu hao** (amortized analysis).

**Space:** `count` có tối đa `k + 1` key ⇒ **`O(k)`** (và không quá số loại ký tự của `s`).

Kiểm chứng: đếm số thao tác thêm/bỏ, so với số bước của Cách 2 trên cùng chuỗi.

In [17]:
def count_ops_window(s, k):
    count, left, adds, removes = {}, 0, 0, 0
    for right, ch in enumerate(s):
        count[ch] = count.get(ch, 0) + 1
        adds += 1
        while len(count) > k:
            count[s[left]] -= 1
            if count[s[left]] == 0:
                del count[s[left]]
            left += 1
            removes += 1
    return adds, removes

import random
random.seed(340)
tests = {"'a'*2000, k=1": ("a" * 2000, 1),
         "'abcd'*500, k=2": ("abcd" * 500, 2),
         "ngẫu nhiên 2000 (a-f), k=3": ("".join(random.choice("abcdef") for _ in range(2000)), 3)}
for name, (s, k) in tests.items():
    adds, removes = count_ops_window(s, k)
    print(f"{name:27}: thêm={adds}, bỏ={removes:4}, tổng={adds + removes:>5} (<= 2n={2 * len(s)}) | Cách 2: {count_steps_quadratic(s, k):>9,} bước")

'a'*2000, k=1              : thêm=2000, bỏ=   0, tổng= 2000 (<= 2n=4000) | Cách 2: 2,001,000 bước
'abcd'*500, k=2            : thêm=2000, bỏ=1998, tổng= 3998 (<= 2n=4000) | Cách 2:     5,997 bước
ngẫu nhiên 2000 (a-f), k=3 : thêm=2000, bỏ=1993, tổng= 3993 (<= 2n=4000) | Cách 2:    11,227 bước


## 9. Cách 4 — Lưu vị trí xuất hiện **cuối**, `left` nhảy một phát

### Cách 3 còn gì để cải tiến?

Ở trace 8.2, tại `right = 6`, Cách 3 phải bỏ **từng ký tự một** (`a`@0, `a`@1, `b`@2, `a`@3). Tổng vẫn `O(n)`, nhưng có hai lý do để muốn cách khác:

1. Ta có thể **tính thẳng** `left` mới thay vì bước từng bước.
2. Cách 3 cần **đọc lại** `s[left]` — tức phải giữ cả chuỗi (hoặc cả cửa sổ) trong bộ nhớ. Nếu ký tự đến dưới dạng **stream** (luồng dữ liệu, đọc một lần rồi mất, không quay lại được), Cách 3 không dùng được trực tiếp.

### Ý tưởng

Lưu `last[ch]` = **vị trí xuất hiện cuối cùng** của `ch` trong cửa sổ.

Khi thêm `s[right]` làm cửa sổ có **`k + 1`** loại, ta phải loại bỏ hẳn **một** loại. Loại nào? Loại có **`last` nhỏ nhất** — gọi là `victim`:

- Để `victim` biến mất, `left` phải vượt qua **lần xuất hiện cuối** của nó ⇒ `left = last[victim] + 1`.
- Mọi loại khác có lần xuất hiện cuối **sau** `last[victim]`, nên vẫn còn trong cửa sổ — ta giữ được cửa sổ **dài nhất** có thể, đúng bằng `L(right)`.
- Chọn loại khác (có `last` lớn hơn) thì `left` nhảy **xa hơn**, cắt cụt cửa sổ và còn để lại trong `last` một key "ma" (ký tự đã nằm ngoài cửa sổ).

Ví dụ `"aabacbebebe"`, `k = 3`, `right = 6`: `last = {a:3, b:5, c:4, e:6}` ⇒ `victim = 'a'`, `left = 3 + 1 = 4` ⇒ cửa sổ `"cbe"`. Đúng bằng Cách 3, chỉ tốn **một** phép gán.

### Bản 4a — `dict` thường + `min`

Tìm `victim` bằng `min(last, key=last.get)` (mục 3.5) tốn `O(k)` mỗi lần.

In [18]:
def longest_k_distinct_lastpos(s: str, k: int) -> int:
    last = {}           # ký tự -> vị trí xuất hiện cuối trong cửa sổ
    left = 0
    best = 0
    for right, ch in enumerate(s):
        last[ch] = right
        if len(last) > k:                       # k+1 loại: loại bỏ đúng 1 loại
            victim = min(last, key=last.get)    # loại có lần xuất hiện cuối sớm nhất
            left = last[victim] + 1
            del last[victim]
        best = max(best, right - left + 1)
    return best

for s, k in [("eceba", 2), ("aa", 1), ("aabacbebebe", 3), ("abc", 0), ("abc", 10)]:
    print(f"s={s!r:14} k={k:2} -> {longest_k_distinct_lastpos(s, k)}")

s='eceba'        k= 2 -> 3
s='aa'           k= 1 -> 2
s='aabacbebebe'  k= 3 -> 7
s='abc'          k= 0 -> 0
s='abc'          k=10 -> 3


Để ý ở đây chỉ cần **`if`**, không cần `while`: mỗi bước chỉ thêm **một** ký tự nên số loại vượt `k` nhiều nhất **1**, và xóa đúng `victim` là đủ đưa về `k` loại.

**Với `k = 0`:** thêm `ch` ⇒ 1 loại > 0 ⇒ `victim = ch`, `left = right + 1`, cửa sổ rỗng ⇒ `best = 0`. Vẫn đúng tự nhiên.

**Độ phức tạp 4a:** `min` duyệt `k + 1` key ⇒ **Time `O(n·k)`**, **Space `O(k)`**. Với `k <= 50` vẫn ổn, nhưng ta làm tốt hơn được.

### Bản 4b — `OrderedDict`: tìm `victim` trong `O(1)`

Nếu giữ các key **sắp theo `last` tăng dần**, `victim` luôn nằm ở **đầu hàng**:

- Gặp `ch`: nếu `ch` đã có thì `move_to_end(ch)` (nó vừa thành "mới nhất"), rồi gán `last[ch] = right`. Ký tự mới thì tự động vào cuối.
- Nếu `len(last) > k`: `victim, pos = last.popitem(last=False)` ⇒ `left = pos + 1`.

Mọi thao tác đều `O(1)` ⇒ **Time `O(n)`**, **Space `O(k)`**. Đây giống hệt ý tưởng của **LRU cache** (bỏ phần tử "lâu không dùng nhất").

In [19]:
from collections import OrderedDict

def longest_k_distinct_ordered(s: str, k: int) -> int:
    last = OrderedDict()   # ký tự -> vị trí cuối, sắp theo vị trí cuối tăng dần
    left = 0
    best = 0
    for right, ch in enumerate(s):
        if ch in last:
            last.move_to_end(ch)               # ch vừa xuất hiện -> mới nhất
        last[ch] = right
        if len(last) > k:
            _, pos = last.popitem(last=False)  # loại có vị trí cuối sớm nhất
            left = pos + 1
        best = max(best, right - left + 1)
    return best

for s, k in [("eceba", 2), ("aa", 1), ("aabacbebebe", 3), ("abc", 0), ("abaccc", 2)]:
    print(f"s={s!r:14} k={k:2} -> {longest_k_distinct_ordered(s, k)}")

s='eceba'        k= 2 -> 3
s='aa'           k= 1 -> 2
s='aabacbebebe'  k= 3 -> 7
s='abc'          k= 0 -> 0
s='abaccc'       k= 2 -> 4


### Trace Cách 4b với `s = "aabacbebebe"`, `k = 3`

Cột `last` ghi các cặp **theo thứ tự trong `OrderedDict`** (đầu hàng bên trái = victim tiềm năng):

| `right` | `ch` | `last` sau cập nhật | Pop? | `left` | Cửa sổ | `best` |
|---|---|---|---|---|---|---|
| 0 | a | [a:0] | | 0 | `a` | 1 |
| 1 | a | [a:1] | | 0 | `aa` | 2 |
| 2 | b | [a:1, b:2] | | 0 | `aab` | 3 |
| 3 | a | [b:2, a:3] ← `a` được chuyển về cuối | | 0 | `aaba` | 4 |
| 4 | c | [b:2, a:3, c:4] | | 0 | `aabac` | 5 |
| 5 | b | [a:3, c:4, b:5] | | 0 | `aabacb` | 6 |
| 6 | e | [a:3, c:4, b:5, e:6] → 4 loại | pop (a, 3) | **4** | `cbe` | 6 |
| 7 | b | [c:4, e:6, b:7] | | 4 | `cbeb` | 6 |
| 8 | e | [c:4, b:7, e:8] | | 4 | `cbebe` | 6 |
| 9 | b | [c:4, e:8, b:9] | | 4 | `cbebeb` | 6 |
| 10 | e | [c:4, b:9, e:10] | | 4 | `cbebebe` | **7** |

So với trace 8.2: cùng `left = 4` tại `right = 6`, nhưng đạt được bằng **một** lần `popitem` thay vì bốn lần bỏ.

In [20]:
def trace_ordered(s, k):
    last, left, best = OrderedDict(), 0, 0
    for right, ch in enumerate(s):
        if ch in last:
            last.move_to_end(ch)
        last[ch] = right
        action = ""
        if len(last) > k:
            victim, pos = last.popitem(last=False)
            left = pos + 1
            action = f" | pop ({victim!r}, {pos}) -> left={left}"
        best = max(best, right - left + 1)
        order = ", ".join(f"{c}:{p}" for c, p in last.items())
        print(f"right={right:2} ch={ch!r} last=[{order}]{action} | cửa sổ {s[left:right + 1]!r}, best={best}")
    return best

print("Kết quả:", trace_ordered("aabacbebebe", 3))

right= 0 ch='a' last=[a:0] | cửa sổ 'a', best=1
right= 1 ch='a' last=[a:1] | cửa sổ 'aa', best=2
right= 2 ch='b' last=[a:1, b:2] | cửa sổ 'aab', best=3
right= 3 ch='a' last=[b:2, a:3] | cửa sổ 'aaba', best=4
right= 4 ch='c' last=[b:2, a:3, c:4] | cửa sổ 'aabac', best=5
right= 5 ch='b' last=[a:3, c:4, b:5] | cửa sổ 'aabacb', best=6
right= 6 ch='e' last=[c:4, b:5, e:6] | pop ('a', 3) -> left=4 | cửa sổ 'cbe', best=6
right= 7 ch='b' last=[c:4, e:6, b:7] | cửa sổ 'cbeb', best=6
right= 8 ch='e' last=[c:4, b:7, e:8] | cửa sổ 'cbebe', best=6
right= 9 ch='b' last=[c:4, e:8, b:9] | cửa sổ 'cbebeb', best=6
right=10 ch='e' last=[c:4, b:9, e:10] | cửa sổ 'cbebebe', best=7
Kết quả: 7


### 9.1. Follow-up phỏng vấn: "Nếu `s` là một **stream** thì sao?"

Câu hỏi phụ hay gặp khi phỏng vấn bài này (không nằm trong đề LeetCode chính thức): ký tự đến **lần lượt**, ta **không lưu được** cả chuỗi (quá lớn), nhưng bất cứ lúc nào cũng phải biết đáp án tính đến hiện tại.

- **Cách 3** cần đọc `s[left]` khi co ⇒ phải nhớ cả cửa sổ, có thể dài tới `n` ⇒ bộ nhớ `O(n)` trong trường hợp xấu.
- **Cách 4b** chỉ cần: `right` (đã đọc bao nhiêu ký tự), `left` (một con số) và `last` (tối đa `k + 1` cặp) ⇒ bộ nhớ **`O(k)`**, mỗi ký tự **`O(1)`**. Không bao giờ phải nhìn lại ký tự cũ!

Ô dưới viết lại Cách 4b nhận một **iterator** bất kỳ (ở đây là generator — hàm có `yield`, sinh từng ký tự một và không thể quay lại), và trả về đáp án "đến giờ" sau mỗi ký tự.

In [21]:
def longest_k_distinct_stream(stream, k):
    """Nhận một iterable ký tự, yield đáp án tính tới ký tự hiện tại. Bộ nhớ O(k)."""
    last = OrderedDict()
    left = 0
    best = 0
    for right, ch in enumerate(stream):     # chỉ đọc mỗi ký tự 1 lần
        if ch in last:
            last.move_to_end(ch)
        last[ch] = right
        if len(last) > k:
            _, pos = last.popitem(last=False)
            left = pos + 1
        best = max(best, right - left + 1)
        yield best

def char_source(text):          # giả lập một luồng dữ liệu: chỉ đọc tới, không quay lại
    for ch in text:
        yield ch

answers = list(longest_k_distinct_stream(char_source("aabacbebebe"), 3))
print("Đáp án sau từng ký tự:", answers)
print("Đáp án cuối:", answers[-1])

Đáp án sau từng ký tự: [1, 2, 3, 4, 5, 6, 6, 6, 6, 6, 7]
Đáp án cuối: 7


## 10. Cách 5 — Cửa sổ "không co" (non-shrinking window)

### Ý tưởng

Đề chỉ hỏi **độ dài lớn nhất**. Vậy khi đã tìm được một cửa sổ dài `M`, ta không quan tâm tới cửa sổ nào ngắn hơn `M` nữa. Mẹo:

- Mỗi bước `right` tăng 1 (cửa sổ dài thêm 1).
- Nếu cửa sổ **vi phạm** (`len(count) > k`), chỉ dịch `left` **đúng 1 bước** bằng `if` (không phải `while`) ⇒ kích thước **giữ nguyên**.
- Như vậy kích thước cửa sổ **không bao giờ giảm**, nó chỉ **tăng** khi gặp cửa sổ hợp lệ dài hơn.
- Cuối cùng đáp án = kích thước cửa sổ = `n - left`.

```text
for right, ch in enumerate(s):
    thêm ch vào count
    if len(count) > k:        # IF, không phải WHILE
        bỏ s[left]; left += 1
return len(s) - left
```

**Hệ quả lạ:** cửa sổ ở giữa quá trình có thể **không hợp lệ** (chứa hơn `k` loại) — và `count` có thể có tới hơn `k + 1` key. Không sao, vì ta chỉ dùng **kích thước** của nó.

In [22]:
def longest_k_distinct_nonshrink(s: str, k: int) -> int:
    count = {}
    left = 0
    for ch in s:
        count[ch] = count.get(ch, 0) + 1
        if len(count) > k:              # chỉ dịch 1 bước, kích thước giữ nguyên
            out = s[left]
            count[out] -= 1
            if count[out] == 0:
                del count[out]
            left += 1
    return len(s) - left                # kích thước cửa sổ cuối = đáp án

for s, k in [("eceba", 2), ("aa", 1), ("aabacbebebe", 3), ("abc", 0), ("abc", 10)]:
    print(f"s={s!r:14} k={k:2} -> {longest_k_distinct_nonshrink(s, k)}")

s='eceba'        k= 2 -> 3
s='aa'           k= 1 -> 2
s='aabacbebebe'  k= 3 -> 7
s='abc'          k= 0 -> 0
s='abc'          k=10 -> 3


### Vì sao mẹo này đúng?

Gọi `M` = kích thước cửa sổ trước bước `right`. Ta chứng minh **bất biến**: *`M` = độ dài lớn nhất của một chuỗi con hợp lệ nằm trong `s[0..right-1]`*.

- Trước bước `right`, cửa sổ là `s[right-M .. right-1]`. Thêm `s[right]` ⇒ cửa sổ thành `s[right-M .. right]`, dài `M + 1`.
- **Nếu nó hợp lệ:** ta giữ nguyên ⇒ kích thước `M + 1`. Đúng, vì vừa tìm thấy một chuỗi con hợp lệ dài `M + 1`, và không thể có chuỗi dài `M + 2` (nếu có, phần bỏ ký tự cuối của nó dài `M + 1` nằm trong `s[0..right-1]`, trái với giả thiết `M` là lớn nhất).
- **Nếu không hợp lệ:** mọi chuỗi con dài `M + 1` kết thúc ở `right` **chính là** cửa sổ này (chỉ có một đoạn dài `M+1` kết thúc tại `right`) ⇒ không có chuỗi hợp lệ dài `M + 1` kết thúc ở `right`; dài hơn càng không (vì chứa đoạn này). Ta dịch `left` 1 bước ⇒ kích thước vẫn `M`. Đúng.

Vậy kết thúc vòng lặp, kích thước `n - left` = độ dài lớn nhất. ∎

Trace dưới đây cho thấy cửa sổ "không hợp lệ" ở `right = 6, 7, 8` nhưng kích thước vẫn giữ 6, rồi hợp lệ trở lại và lớn lên thành 7.

In [23]:
def trace_nonshrink(s, k):
    count, left = {}, 0
    for right, ch in enumerate(s):
        count[ch] = count.get(ch, 0) + 1
        moved = ""
        if len(count) > k:
            out = s[left]
            count[out] -= 1
            if count[out] == 0:
                del count[out]
            left += 1
            moved = f"bỏ {out}@{left - 1}"
        status = "hợp lệ    " if len(count) <= k else "KHÔNG hợp lệ"
        print(f"right={right:2} ch={ch!r} {moved:7} | [{left}..{right}] {s[left:right + 1]!r:10} kích thước {right - left + 1} | {len(count)} loại, {status}")
    print("Đáp án = n - left =", len(s), "-", left, "=", len(s) - left)

trace_nonshrink("aabacbebebe", 3)

right= 0 ch='a'         | [0..0] 'a'        kích thước 1 | 1 loại, hợp lệ    
right= 1 ch='a'         | [0..1] 'aa'       kích thước 2 | 1 loại, hợp lệ    
right= 2 ch='b'         | [0..2] 'aab'      kích thước 3 | 2 loại, hợp lệ    
right= 3 ch='a'         | [0..3] 'aaba'     kích thước 4 | 2 loại, hợp lệ    
right= 4 ch='c'         | [0..4] 'aabac'    kích thước 5 | 3 loại, hợp lệ    
right= 5 ch='b'         | [0..5] 'aabacb'   kích thước 6 | 3 loại, hợp lệ    
right= 6 ch='e' bỏ a@0  | [1..6] 'abacbe'   kích thước 6 | 4 loại, KHÔNG hợp lệ
right= 7 ch='b' bỏ a@1  | [2..7] 'bacbeb'   kích thước 6 | 4 loại, KHÔNG hợp lệ
right= 8 ch='e' bỏ b@2  | [3..8] 'acbebe'   kích thước 6 | 4 loại, KHÔNG hợp lệ
right= 9 ch='b' bỏ a@3  | [4..9] 'cbebeb'   kích thước 6 | 3 loại, hợp lệ    
right=10 ch='e'         | [4..10] 'cbebebe'  kích thước 7 | 3 loại, hợp lệ    
Đáp án = n - left = 11 - 4 = 7


Hai lưu ý thực tế về Cách 5:

1. **Cửa sổ cuối không chắc là đáp án.** Ở ví dụ trên tình cờ cửa sổ cuối `"cbebebe"` hợp lệ. Nhưng với `s = "aab"`, `k = 1`: kích thước cuối là 2 (đúng — đáp án là `"aa"`), còn cửa sổ cuối lại là `"ab"` — **không hợp lệ**! Muốn lấy **chính chuỗi con** thì phải ghi lại vị trí mỗi lần cửa sổ lớn lên (ô code ngay dưới minh họa).
2. **Time `O(n)`** (mỗi bước làm `O(1)`), **Space `O(số loại trong cửa sổ)`** — có thể nhiều hơn `k + 1` vì cửa sổ không hợp lệ được phép chứa nhiều loại, nhưng không quá số loại của `s`.

In [24]:
s, k = "aab", 1
left_final = len(s) - longest_k_distinct_nonshrink(s, k)
print("Kích thước:", longest_k_distinct_nonshrink(s, k), "| cửa sổ cuối:", repr(s[left_final:]),
      "| số loại:", len(set(s[left_final:])), "> k" if len(set(s[left_final:])) > k else "")

Kích thước: 2 | cửa sổ cuối: 'ab' | số loại: 2 > k


## 11. Code nộp LeetCode

Chuyển Cách 3 vào đúng khung `class Solution`: tên method `lengthOfLongestSubstringKDistinct`, tham số `self, s: str, k: int`, trả về `int`. Code nộp **không in debug**. Dòng `if k == 0` là tùy chọn (thuật toán vẫn đúng khi bỏ), giữ lại để người đọc thấy rõ edge case.

Ô thứ hai là bản nộp thay thế dùng Cách 4b (`OrderedDict`) — hữu ích nếu người phỏng vấn hỏi tiếp về stream. Đặt tên class khác để không ghi đè `Solution` trong notebook; khi nộp thì đổi lại thành `Solution`.

In [25]:
class Solution:
    def lengthOfLongestSubstringKDistinct(self, s: str, k: int) -> int:
        if k == 0:
            return 0
        count = {}
        left = 0
        best = 0
        for right, ch in enumerate(s):
            count[ch] = count.get(ch, 0) + 1
            while len(count) > k:
                out = s[left]
                count[out] -= 1
                if count[out] == 0:
                    del count[out]
                left += 1
            best = max(best, right - left + 1)
        return best


sol = Solution()
assert sol.lengthOfLongestSubstringKDistinct("eceba", 2) == 3        # Example 1
assert sol.lengthOfLongestSubstringKDistinct("aa", 1) == 2           # Example 2
assert sol.lengthOfLongestSubstringKDistinct("a", 0) == 0            # k = 0
assert sol.lengthOfLongestSubstringKDistinct("abc", 0) == 0          # k = 0, nhiều ký tự
assert sol.lengthOfLongestSubstringKDistinct("a", 1) == 1            # 1 ký tự
assert sol.lengthOfLongestSubstringKDistinct("abc", 50) == 3         # k lớn hơn số loại
assert sol.lengthOfLongestSubstringKDistinct("aabbbc", 1) == 3       # k = 1: đoạn toàn 1 ký tự
assert sol.lengthOfLongestSubstringKDistinct("aabacbebebe", 3) == 7  # co nhiều bước
assert sol.lengthOfLongestSubstringKDistinct("aA", 1) == 1           # hoa/thường khác nhau
assert sol.lengthOfLongestSubstringKDistinct("ab ab!", 3) == 5       # ký tự bất kỳ
print("Tất cả test Solution (Cách 3) PASS")

Tất cả test Solution (Cách 3) PASS


In [26]:
from collections import OrderedDict

class SolutionOrderedDict:
    def lengthOfLongestSubstringKDistinct(self, s: str, k: int) -> int:
        last = OrderedDict()
        left = 0
        best = 0
        for right, ch in enumerate(s):
            if ch in last:
                last.move_to_end(ch)
            last[ch] = right
            if len(last) > k:
                _, pos = last.popitem(last=False)
                left = pos + 1
            best = max(best, right - left + 1)
        return best


sol2 = SolutionOrderedDict()
for s, k, expected in [("eceba", 2, 3), ("aa", 1, 2), ("a", 0, 0), ("abc", 50, 3),
                       ("aabacbebebe", 3, 7), ("abaccc", 2, 4), ("aA", 1, 1)]:
    assert sol2.lengthOfLongestSubstringKDistinct(s, k) == expected, (s, k)
print("Tất cả test SolutionOrderedDict (Cách 4b) PASS")

Tất cả test SolutionOrderedDict (Cách 4b) PASS


### Test đối chiếu chéo trên input ngẫu nhiên

Mọi cách phải cho **cùng** kết quả với brute force (Cách 1, "trọng tài") trên hàng nghìn chuỗi ngẫu nhiên nhỏ. Dùng bảng chữ cái nhỏ (`"abcd"`) để chuỗi có nhiều ký tự trùng — đó là nơi lỗi hay ẩn nấp — và cho `k` chạy từ `0` tới `5` (kể cả `k` lớn hơn số loại). `random.seed(340)` cố định để lần chạy nào cũng giống nhau.

Ô tiếp theo chạy ở **giới hạn của đề** `n = 5 * 10^4`: chuỗi ngẫu nhiên từ 52 chữ cái hoa/thường, vài giá trị `k`. (Thời gian cụ thể tùy máy; điều đáng xem là các cách nhanh đều xong trong khoảng vài chục mili-giây và cho cùng kết quả. Cách 4a tốn thêm `O(k)` cho `min` **mỗi lần phải loại một ký tự**, nên nó chỉ thua rõ khi `k` lớn **và** phải loại nhiều lần; với `k <= 50` chênh lệch nhỏ.)

In [27]:
import random
random.seed(340)

methods = {
    "quadratic": longest_k_distinct_quadratic,
    "window": longest_k_distinct_window,
    "lastpos": longest_k_distinct_lastpos,
    "ordered": longest_k_distinct_ordered,
    "stream": lambda s, k: (list(longest_k_distinct_stream(iter(s), k)) or [0])[-1],
    "nonshrink": longest_k_distinct_nonshrink,
    "Solution": Solution().lengthOfLongestSubstringKDistinct,
    "SolutionOD": SolutionOrderedDict().lengthOfLongestSubstringKDistinct,
}
n_tests = 0
for _ in range(3000):
    n = random.randint(1, 12)
    s = "".join(random.choice("abcd") for _ in range(n))
    k = random.randint(0, 5)
    expected = longest_k_distinct_bruteforce(s, k)
    for name, f in methods.items():
        got = f(s, k)
        assert got == expected, (name, s, k, got, expected)
    n_tests += 1
print(f"{n_tests} chuỗi ngẫu nhiên x {len(methods)} cách: tất cả khớp brute force")

3000 chuỗi ngẫu nhiên x 8 cách: tất cả khớp brute force


In [28]:
import string, time
random.seed(2024)
big = "".join(random.choice(string.ascii_letters) for _ in range(5 * 10**4))
for k in [1, 10, 50]:
    row = []
    results = set()
    for name in ["window", "lastpos", "ordered", "nonshrink"]:
        t0 = time.perf_counter()
        results.add(methods[name](big, k))
        row.append(f"{name}={1000 * (time.perf_counter() - t0):6.1f}ms")
    assert len(results) == 1
    print(f"k={k:2}: đáp án {results.pop():3} | " + "  ".join(row))

k= 1: đáp án   4 | window=   8.2ms  lastpos=  11.1ms  ordered=   9.0ms  nonshrink=   8.9ms


k=10: đáp án  19 | window=  10.1ms  lastpos=  23.2ms  ordered=  10.3ms  nonshrink=   6.8ms


k=50: đáp án 369 | window=  11.3ms  lastpos=   8.2ms  ordered=   6.7ms  nonshrink=   9.4ms


## 12. Vì sao Cách 3 đúng?

**Bất biến (invariant)** — sau bước "co" của mỗi vòng `right`:

1. `count` đếm **chính xác** các ký tự của `s[left..right]`, và không có key nào bằng 0 ⇒ `len(count)` = số loại trong cửa sổ.
2. `len(count) <= k` — cửa sổ **hợp lệ**.
3. `left = L(right)` — `left` là đầu trái **nhỏ nhất** có thể cho `right` này.

**Vì sao (3) đúng?** Theo mục 6, `L(right) >= L(right-1)`. Vòng trước ta có `left = L(right-1)`, tức `left <= L(right)`. Vòng `while` tăng `left` từng bước và **dừng ngay** khi cửa sổ hợp lệ lần đầu. Mọi `left' < L(right)` đều cho cửa sổ không hợp lệ (định nghĩa của `L`), nên `while` không dừng ở đó; tới `left = L(right)` thì hợp lệ nên dừng. Vậy `left = L(right)` — `while` **không co quá tay**, cũng **không co thiếu**.

**Vì sao không bỏ sót?** Đáp án thật là một đoạn `s[i..j]` nào đó. Khi `right = j`, cửa sổ ta xét là `s[L(j)..j]` với `L(j) <= i`, nên độ dài ta ghi nhận `>= j - i + 1`. Mà mọi cửa sổ ta ghi nhận đều hợp lệ ⇒ không vượt đáp án thật. Vậy `best` = đáp án. ∎

Cách 4 đúng vì cùng duy trì `left = L(right)`, chỉ tính bằng công thức `last[victim] + 1` thay vì bước từng bước; Cách 5 đúng theo chứng minh riêng ở mục 10.

## 13. So sánh các cách

Ký hiệu: `n = len(s)` (tối đa `5 * 10^4`), `k` (tối đa 50).

| Cách | Ý tưởng | Time | Space | Ưu điểm | Nhược điểm |
|---|---|---|---|---|---|
| 1. Brute force | Mọi `s[i..j]` + `set` | `O(n³)` | `O(n)` | Cực dễ viết, làm "trọng tài" test | Quá chậm |
| 2. Cố định trái | Mở rộng + `break` khi vượt `k` | `O(n²)` | `O(k)` | Vẫn trực giác, nhanh hơn nhiều | TLE với `n = 5·10^4` (ví dụ `"a"*n`) |
| 3. Window + đếm | `while len(count) > k` co | `O(n)` | `O(k)` | **Chuẩn, dễ tổng quát** cho cả họ bài | Phải đọc lại `s[left]` |
| 4a. Vị trí cuối + `min` | `left = min(last) + 1` | `O(n·k)` | `O(k)` | Nhảy một phát, không đọc lại `s` | `min` tốn `O(k)` |
| 4b. `OrderedDict` | Pop đầu hàng | `O(n)` | `O(k)` | `O(1)`/ký tự, **hợp với stream** | Phải nhớ `move_to_end` |
| 5. Không co | `if` thay `while`, đáp án `n - left` | `O(n)` | `O(số loại)` | Code ngắn, mẹo hay | Cửa sổ có thể không hợp lệ; khó lấy chuỗi con |

Khi phỏng vấn: trình bày Cách 1/2 trong 1 phút để cho thấy bạn hiểu bài, viết **Cách 3**, rồi nhắc Cách 4b khi được hỏi về stream.

## 14. Lỗi thường gặp

### Lỗi 1 — Sai ở dòng `best`: ghi nhận **trước** khi co, hoặc khởi tạo `best = 1`

```python
count[ch] = count.get(ch, 0) + 1
best = max(best, right - left + 1)   # SAI: cửa sổ có thể đang có k+1 loại
while len(count) > k: ...
```

- **Ghi nhận quá sớm:** ở `"eceba"`, `k = 2`, tại `right = 3` cửa sổ `"eceb"` (3 loại) bị ghi nhận độ dài 4. Với `k = 0` còn tệ hơn: mọi ký tự đơn đều bị ghi nhận. **Sửa:** luôn theo thứ tự **thêm → co → ghi nhận**.
- **`best = 1` "vì `s.length >= 1`"** nghe hợp lý nhưng **sai khi `k = 0`** (đáp án phải là 0). **Sửa:** khởi tạo `best = 0`.

In [29]:
def window_record_too_early(s, k):           # BẢN SAI 1
    count, left, best = {}, 0, 0
    for right, ch in enumerate(s):
        count[ch] = count.get(ch, 0) + 1
        best = max(best, right - left + 1)   # ghi nhận quá sớm
        while len(count) > k:
            count[s[left]] -= 1
            if count[s[left]] == 0:
                del count[s[left]]
            left += 1
    return best

def window_best_one(s, k):                   # BẢN SAI 2
    count, left, best = {}, 0, 1             # best = 1 -> sai khi k = 0
    for right, ch in enumerate(s):
        count[ch] = count.get(ch, 0) + 1
        while len(count) > k:
            count[s[left]] -= 1
            if count[s[left]] == 0:
                del count[s[left]]
            left += 1
        best = max(best, right - left + 1)
    return best

print("Ghi nhận sớm:", window_record_too_early("eceba", 2), "| Đúng:", longest_k_distinct_window("eceba", 2))
print("Ghi nhận sớm:", window_record_too_early("abc", 0), "| Đúng:", longest_k_distinct_window("abc", 0))
print("best = 1    :", window_best_one("abc", 0), "| Đúng:", longest_k_distinct_window("abc", 0))

Ghi nhận sớm: 4 | Đúng: 3
Ghi nhận sớm: 1 | Đúng: 0
best = 1    : 1 | Đúng: 0


### Lỗi 2 — Dùng `Counter` (hoặc `dict`) mà **không xóa** key về 0

```python
count = Counter()
...
while len(count) > k:
    count[s[left]] -= 1       # SAI: key về 0 vẫn còn, len(count) không giảm
    left += 1
```

**Vì sao sai:** `len(count)` không bao giờ giảm ⇒ `while` chạy mãi, `left` vượt qua `right` rồi vượt cả cuối chuỗi ⇒ `IndexError` (hoặc đáp án sai). **Sửa:** `if count[x] == 0: del count[x]` — hoặc đếm riêng một biến `distinct`.

In [30]:
from collections import Counter

def window_counter_no_del(s, k):             # BẢN SAI
    count, left, best = Counter(), 0, 0
    for right, ch in enumerate(s):
        count[ch] += 1
        while len(count) > k:
            count[s[left]] -= 1               # quên del
            left += 1
        best = max(best, right - left + 1)
    return best

try:
    print(window_counter_no_del("eceba", 2))
except IndexError as e:
    print("Bản sai bị lỗi:", type(e).__name__, "-", e)
print("Bản đúng:", longest_k_distinct_window("eceba", 2))

Bản sai bị lỗi: IndexError - string index out of range
Bản đúng: 3


### Lỗi 3 — Cách 4 (vị trí cuối): quên `move_to_end`, quên `+ 1`, hoặc loại nhầm `s[left]`

- **4b quên `move_to_end`:** gán `last[ch] = right` cho key đã có chỉ đổi value, **không** đổi vị trí ⇒ `OrderedDict` sắp theo lần xuất hiện **đầu tiên**. Với `"abaccc"`, `k = 2`: tại `right = 3` (`c`), đầu hàng là `a` (dù `a` vừa xuất hiện ở vị trí 2) ⇒ pop `a`, `left = 3`, mất cửa sổ đúng `"accc"` ⇒ trả về 3 thay vì 4.
- **4a quên `+ 1`** (`left = last[victim]`) ⇒ cửa sổ vẫn còn **một** `victim` ⇒ có `k + 1` loại mà vẫn được ghi nhận. Với `"abc"`, `k = 2` trả về 3 thay vì 2.
- **4a chọn `victim = s[left]`** ⇒ ký tự ở đầu trái có thể còn xuất hiện **sau**; `left` nhảy quá xa, còn loại có `last` nhỏ hơn bị bỏ lại trong `last` như "bóng ma". Với `"abaccc"`, `k = 2` trả về 3 thay vì 4.

**Sửa:** `if ch in last: last.move_to_end(ch)`; `victim` = loại có `last` **nhỏ nhất**; `left = last[victim] + 1`.

In [31]:
def ordered_no_move_to_end(s, k):            # BẢN SAI A
    last, left, best = OrderedDict(), 0, 0
    for right, ch in enumerate(s):
        last[ch] = right                     # thiếu move_to_end
        if len(last) > k:
            _, pos = last.popitem(last=False)
            left = pos + 1
        best = max(best, right - left + 1)
    return best

def lastpos_buggy(s, k, plus_one=True, victim_is_left=False):   # BẢN SAI B / C
    last, left, best = {}, 0, 0
    for right, ch in enumerate(s):
        last[ch] = right
        if len(last) > k:
            victim = s[left] if victim_is_left else min(last, key=last.get)
            left = last[victim] + (1 if plus_one else 0)
            del last[victim]
        best = max(best, right - left + 1)
    return best

print("Quên move_to_end:", ordered_no_move_to_end("abaccc", 2), "| Đúng:", longest_k_distinct_ordered("abaccc", 2))
print("Thiếu +1        :", lastpos_buggy("abc", 2, plus_one=False), "| Đúng:", longest_k_distinct_lastpos("abc", 2))
print("Victim = s[left]:", lastpos_buggy("abaccc", 2, victim_is_left=True), "| Đúng:", longest_k_distinct_lastpos("abaccc", 2))

Quên move_to_end: 3 | Đúng: 4
Thiếu +1        : 3 | Đúng: 2
Victim = s[left]: 3 | Đúng: 4


### Lỗi 4 — Các lỗi "lặt vặt"

- Quên `+ 1` trong `right - left + 1` (độ dài đoạn lấy cả hai đầu).
- Dùng `min(last)` thay vì `min(last, key=last.get)` — so **key** (chữ cái) thay vì so **value** (vị trí), mục 3.5.
- Dùng `if` thay `while` trong Cách 3 mà vẫn ghi nhận `best` như cũ thì lại **vô tình thành Cách 5** và vẫn ra đúng độ dài — nhưng nếu dùng cửa sổ đó để **trả về chuỗi con** thì sai (mục 10, ví dụ `"aab"`).
- Tự "tối ưu" bằng `if k >= len(set(s)): return len(s)` thì được, nhưng `len(set(s))` tốn `O(n)` — không sai, chỉ thừa.
- Nhầm `s.length` thành giới hạn của `k`: đề cho `k <= 50`, còn `n <= 5 * 10^4`; `k` có thể **lớn hơn** số loại ký tự của `s`.

## 15. Mẹo nhận dạng pattern

Nghĩ ngay tới **sliding window kích thước thay đổi** khi đề có đủ các dấu hiệu:

1. Hỏi về **substring / subarray** (đoạn **liền nhau**), không phải subsequence.
2. Tìm đoạn **dài nhất** (hoặc ngắn nhất, hoặc đếm số đoạn) thỏa một điều kiện.
3. Điều kiện có tính **"co lại vẫn đúng"** (monotone): đoạn hợp lệ thì mọi đoạn con cũng hợp lệ. Ví dụ "tối đa `k` loại", "tổng ≤ S với số không âm", "không lặp ký tự", "tối đa `k` số 0".
4. Kiểm tra điều kiện được **cập nhật dần** khi thêm/bỏ một phần tử (dùng `dict` đếm, biến tổng, biến đếm...).

Khung nhớ: **thêm phải → `while` vi phạm: bỏ trái → ghi nhận**.

Dấu hiệu **không** dùng được trực tiếp: điều kiện kiểu "**ít nhất** `k` lần mỗi ký tự" (LeetCode 395) — co lại có thể làm đoạn từ hợp lệ thành không hợp lệ rồi lại hợp lệ, mất tính đơn điệu. Khi đó phải đổi bài (ví dụ cố định số loại ký tự rồi mới trượt).

## 16. Biến thể và bài liên quan

| Bài | Liên hệ với 340 |
|---|---|
| **159** — At Most Two Distinct Characters | Chính là 340 với `k = 2` (có notebook riêng). |
| **904** — Fruit Into Baskets | 340 với `k = 2` trên **mảng số** (mỗi số là một loại trái cây). |
| **3** — Longest Substring Without Repeating Characters | Cùng khung, điều kiện đổi thành "không ký tự nào đếm > 1". |
| **992** — Subarrays with K Different Integers | **Đếm** số đoạn có **đúng** `K` loại = `atMost(K) - atMost(K-1)`. |
| **1004** — Max Consecutive Ones III | Cửa sổ chứa tối đa `k` số 0. |
| **424** — Longest Repeating Character Replacement | Cửa sổ hợp lệ khi `độ dài - tần suất lớn nhất <= k`; hay dùng mẹo "không co" của Cách 5. |
| **395** — At Least K Repeating Characters | Trông giống nhưng **không** đơn điệu (xem mục 15). |

Vì hàm của ta chỉ dùng `enumerate`, chỉ số `[left]` và `dict`, nó chạy được trên **list** cũng như chuỗi. Giải luôn 159 và 904 bằng chính hàm Cách 3:

In [32]:
def length_of_longest_two_distinct(s):          # LeetCode 159
    return longest_k_distinct_window(s, 2)

def total_fruit(fruits):                        # LeetCode 904
    return longest_k_distinct_window(fruits, 2)

assert length_of_longest_two_distinct("eceba") == 3
assert length_of_longest_two_distinct("ccaabbb") == 5
assert total_fruit([1, 2, 1]) == 3
assert total_fruit([0, 1, 2, 2]) == 3
assert total_fruit([1, 2, 3, 2, 2]) == 4
assert total_fruit([3, 3, 3, 1, 2, 1, 1, 2, 3, 3, 4]) == 5
print("159 và 904 giải bằng cùng một hàm: PASS")

159 và 904 giải bằng cùng một hàm: PASS


## 17. Bài tập tự luyện

Làm theo thứ tự, **tự làm trước** rồi mới xem gợi ý (mục 18) và đáp án (mục 19). Các ô TODO chỉ định nghĩa hàm — test để ở dạng comment, làm xong thì bỏ `#` để chạy.

### Bài 1 — Đoán output (không chạy code)

Với `f = longest_k_distinct_window`, đoán kết quả:

```python
f("aaabbbccc", 2)
f("abc", 0)
f("abc", 10)
f("abaccc", 1)
f("ababffzzeee", 3)
```

### Bài 2 — Tính tay Cách 4b (`OrderedDict`)

Với `s = "abcbbcad"`, `k = 2`, lập bảng như mục 9: mỗi `right` ghi `last` (theo thứ tự trong `OrderedDict`), có pop không, `left`, cửa sổ và `best`. Đáp án cuối là bao nhiêu? Ở `right = 6`, ký tự nào bị pop và vì sao **không phải** `c`?

### Bài 3 — Điền TODO: trả về **chính chuỗi con**

Viết `longest_k_distinct_substring(s, k)` trả về chuỗi con dài nhất có tối đa `k` loại. Nếu có nhiều đáp án cùng độ dài, trả về chuỗi **xuất hiện sớm nhất** (bên trái nhất). Với `k = 0` trả về `""`.

In [33]:
def longest_k_distinct_substring(s: str, k: int) -> str:
    count = {}
    left = 0
    best_len, best_start = 0, 0
    for right, ch in enumerate(s):
        # TODO 1: thêm ch vào count
        # TODO 2: while còn hơn k loại: bỏ s[left] (nhớ del khi về 0), left += 1
        # TODO 3: nếu cửa sổ hiện tại DÀI HƠN HẲN best_len thì cập nhật best_len, best_start
        pass
    return s[best_start:best_start + best_len]

# Tự làm xong rồi mới mở test:
# assert longest_k_distinct_substring("eceba", 2) == "ece"
# assert longest_k_distinct_substring("aabacbebebe", 3) == "cbebebe"
# assert longest_k_distinct_substring("abcd", 2) == "ab"     # nhiều đáp án dài 2 -> lấy bên trái nhất
# assert longest_k_distinct_substring("abc", 0) == ""

### Bài 4 — Tự viết lại: Cách 4 với `dict` thường nhưng `O(1)` mỗi bước

Không dùng `OrderedDict` và không dùng `min`. Gợi ý cấu trúc: `dict` thường nhớ **thứ tự chèn**; nếu mỗi lần gặp `ch` ta **xóa rồi chèn lại** `ch`, thì key đầu tiên của `dict` luôn là ký tự có vị trí cuối sớm nhất.

In [34]:
def longest_k_distinct_dict_order(s: str, k: int) -> int:
    # TODO: viết lại Cách 4 chỉ với dict thường, mỗi bước O(1)
    raise NotImplementedError

# Tự làm xong rồi mới mở test:
# for s, k, exp in [("eceba", 2, 3), ("aa", 1, 2), ("abc", 0, 0), ("abaccc", 2, 4), ("aabacbebebe", 3, 7)]:
#     assert longest_k_distinct_dict_order(s, k) == exp, (s, k)

### Bài 5 — Biến thể: **đúng** `k` loại

Viết `longest_exactly_k_distinct(s, k)` (giả sử `k >= 1`) trả về độ dài chuỗi con dài nhất có **đúng** `k` loại ký tự; nếu không có thì trả về `-1`. Ví dụ `"aabacbebebe"`, `k = 3` → `7`; `"aaaa"`, `k = 2` → `-1`.

### Bài 6 — Mini challenge: **đếm** số chuỗi con có đúng `k` loại

Viết `count_exactly_k_distinct(s, k)` đếm số chuỗi con (khác rỗng, tính theo vị trí) có **đúng** `k` loại ký tự — phiên bản chuỗi của LeetCode 992. Yêu cầu `O(n)`. Ví dụ `s = "abab"`, `k = 2` → `6` (`"ab"`, `"ba"`, `"ab"`, `"aba"`, `"bab"`, `"abab"`).

Hai bài dùng chung một ô TODO bên dưới.

In [35]:
# ===== Bài 5 =====
def longest_exactly_k_distinct(s: str, k: int) -> int:
    # TODO: dùng lại khung Cách 3, chỉ đổi chỗ "ghi nhận"
    pass

# Tự làm xong rồi mới mở test:
# assert longest_exactly_k_distinct("aabacbebebe", 3) == 7
# assert longest_exactly_k_distinct("aaaa", 2) == -1
# assert longest_exactly_k_distinct("aaabbb", 1) == 3
# assert longest_exactly_k_distinct("eceba", 4) == 5


# ===== Bài 6 =====
def count_exactly_k_distinct(s: str, k: int) -> int:
    # TODO: viết hàm phụ đếm số chuỗi con có TỐI ĐA k loại, rồi kết hợp
    pass

# Tự làm xong rồi mới mở test:
# assert count_exactly_k_distinct("abab", 2) == 6
# assert count_exactly_k_distinct("aaa", 1) == 6
# assert count_exactly_k_distinct("abc", 0) == 0
# assert count_exactly_k_distinct("abc", 3) == 1

## 18. Gợi ý (mở dần từng mức)

**Bài 1.** (1) Với mỗi câu, hỏi: đoạn nào dài nhất có ≤ k loại? (2) `k = 0` và `k` lớn hơn số loại là hai edge case của mục 2.3. (3) Câu cuối: thử các đoạn chứa `ff`, `zz`, `eee` liền nhau.

**Bài 2.** (1) Nhớ: gặp lại ký tự cũ thì nó chuyển về **cuối** hàng. (2) Pop xảy ra khi có 3 loại; pop luôn lấy **đầu** hàng. (3) Ở `right = 5`, `c` vừa được chuyển về cuối, nên đầu hàng là `b` (vị trí cuối 4).

**Bài 3.** (1) Giống hệt Cách 3, chỉ cần nhớ thêm **vị trí bắt đầu** của kỷ lục. (2) Dùng `>` (không phải `>=`) để giữ đáp án bên trái nhất. (3) `if right - left + 1 > best_len: best_len, best_start = right - left + 1, left`.

**Bài 4.** (1) `dict` thường giữ thứ tự chèn; `next(iter(d))` cho key đầu tiên trong `O(1)`. (2) Gặp `ch` đã có: `del last[ch]` rồi `last[ch] = right` ⇒ `ch` thành key cuối. (3) Khi `len(last) > k`: `victim = next(iter(last))`, `left = last.pop(victim) + 1`.

**Bài 5.** (1) Chuỗi dài nhất có **đúng** k loại kết thúc tại `right`, nếu tồn tại, chính là cửa sổ "tối đa k" `s[L(right)..right]`. Vì sao? (2) Vậy chỉ ghi nhận khi `len(count) == k`. (3) Cuối cùng `return best if best > 0 else -1`.

**Bài 6.** (1) Với mỗi `right`, số chuỗi con có **tối đa** k loại kết thúc tại `right` là `right - left + 1` (mọi đầu trái từ `left` tới `right`). (2) "Đúng k" = "tối đa k" − "tối đa k−1". (3) Cẩn thận `k - 1 = -1`: hàm "tối đa" phải trả `0` khi `k < 0` (nếu không, `while len(count) > -1` sẽ chạy quá chuỗi).

## 19. Đáp án tham khảo

### Đáp án bài 1

| Lời gọi | Kết quả | Giải thích |
|---|---|---|
| `f("aaabbbccc", 2)` | 6 | `"aaabbb"` hoặc `"bbbccc"` |
| `f("abc", 0)` | 0 | `k = 0` |
| `f("abc", 10)` | 3 | `k` ≥ số loại ⇒ cả chuỗi |
| `f("abaccc", 1)` | 3 | `"ccc"` |
| `f("ababffzzeee", 3)` | 7 | `"ffzzeee"` (f, z, e) |

### Đáp án bài 2 — `s = "abcbbcad"`, `k = 2`

| `right` | `ch` | `last` sau cập nhật | Pop? | `left` | Cửa sổ | `best` |
|---|---|---|---|---|---|---|
| 0 | a | [a:0] | | 0 | `a` | 1 |
| 1 | b | [a:0, b:1] | | 0 | `ab` | 2 |
| 2 | c | [a:0, b:1, c:2] → 3 loại | pop (a, 0) | 1 | `bc` | 2 |
| 3 | b | [c:2, b:3] | | 1 | `bcb` | 3 |
| 4 | b | [c:2, b:4] | | 1 | `bcbb` | 4 |
| 5 | c | [b:4, c:5] ← `c` về cuối | | 1 | `bcbbc` | **5** |
| 6 | a | [b:4, c:5, a:6] → 3 loại | pop (b, 4) | 5 | `ca` | 5 |
| 7 | d | [c:5, a:6, d:7] → 3 loại | pop (c, 5) | 6 | `ad` | 5 |

Đáp án **5** (`"bcbbc"`). Ở `right = 6`, `b` bị pop vì lần cuối của `b` (vị trí 4) **sớm hơn** lần cuối của `c` (vị trí 5): bỏ `b` chỉ cần `left = 5`, giữ lại được `c`@5. Nếu bỏ `c` thì phải đặt `left = 6` và cửa sổ chỉ còn `"a"`.

Ô dưới kiểm chứng đáp án bài 1 và in trace bài 2. Các ô tiếp theo là **đáp án bài 3 → 6**, mỗi hàm có assert ví dụ và test đối chiếu với brute force trên chuỗi ngẫu nhiên.

In [36]:
f = longest_k_distinct_window
answers = [f("aaabbbccc", 2), f("abc", 0), f("abc", 10), f("abaccc", 1), f("ababffzzeee", 3)]
print("Bài 1:", answers)
assert answers == [6, 0, 3, 3, 7]
print("Bài 2 — kết quả:", trace_ordered("abcbbcad", 2))

Bài 1: [6, 0, 3, 3, 7]
right= 0 ch='a' last=[a:0] | cửa sổ 'a', best=1
right= 1 ch='b' last=[a:0, b:1] | cửa sổ 'ab', best=2
right= 2 ch='c' last=[b:1, c:2] | pop ('a', 0) -> left=1 | cửa sổ 'bc', best=2
right= 3 ch='b' last=[c:2, b:3] | cửa sổ 'bcb', best=3
right= 4 ch='b' last=[c:2, b:4] | cửa sổ 'bcbb', best=4
right= 5 ch='c' last=[b:4, c:5] | cửa sổ 'bcbbc', best=5
right= 6 ch='a' last=[c:5, a:6] | pop ('b', 4) -> left=5 | cửa sổ 'ca', best=5
right= 7 ch='d' last=[a:6, d:7] | pop ('c', 5) -> left=6 | cửa sổ 'ad', best=5
Bài 2 — kết quả: 5


In [37]:
# ---- Bài 3 ----
def longest_k_distinct_substring(s: str, k: int) -> str:
    count = {}
    left = 0
    best_len, best_start = 0, 0
    for right, ch in enumerate(s):
        count[ch] = count.get(ch, 0) + 1
        while len(count) > k:
            out = s[left]
            count[out] -= 1
            if count[out] == 0:
                del count[out]
            left += 1
        if right - left + 1 > best_len:          # '>' giữ đáp án bên trái nhất
            best_len, best_start = right - left + 1, left
    return s[best_start:best_start + best_len]

assert longest_k_distinct_substring("eceba", 2) == "ece"
assert longest_k_distinct_substring("aabacbebebe", 3) == "cbebebe"
assert longest_k_distinct_substring("abcd", 2) == "ab"
assert longest_k_distinct_substring("abc", 0) == ""

def substring_bruteforce(s, k):                  # chuỗi dài nhất, bên trái nhất
    best = ""
    for i in range(len(s)):
        for j in range(i, len(s)):
            if len(set(s[i:j + 1])) <= k and j - i + 1 > len(best):
                best = s[i:j + 1]
    return best

random.seed(3)
for _ in range(1000):
    s = "".join(random.choice("abc") for _ in range(random.randint(1, 10)))
    k = random.randint(0, 4)
    assert longest_k_distinct_substring(s, k) == substring_bruteforce(s, k), (s, k)
print("Bài 3 PASS")

Bài 3 PASS


In [38]:
# ---- Bài 4 ----
def longest_k_distinct_dict_order(s: str, k: int) -> int:
    last = {}            # dict thường: thứ tự chèn = thứ tự vị trí cuối tăng dần
    left = 0
    best = 0
    for right, ch in enumerate(s):
        if ch in last:
            del last[ch]                 # xóa rồi chèn lại -> ch thành key cuối
        last[ch] = right
        if len(last) > k:
            victim = next(iter(last))    # key đầu tiên = vị trí cuối sớm nhất, O(1)
            left = last.pop(victim) + 1
        best = max(best, right - left + 1)
    return best

for s, k, exp in [("eceba", 2, 3), ("aa", 1, 2), ("abc", 0, 0), ("abaccc", 2, 4), ("aabacbebebe", 3, 7)]:
    assert longest_k_distinct_dict_order(s, k) == exp, (s, k)

random.seed(4)
for _ in range(1000):
    s = "".join(random.choice("abcd") for _ in range(random.randint(1, 12)))
    k = random.randint(0, 5)
    assert longest_k_distinct_dict_order(s, k) == longest_k_distinct_bruteforce(s, k), (s, k)
print("Bài 4 PASS")

Bài 4 PASS


**Vì sao Bài 5 chỉ cần ghi nhận khi `len(count) == k`?** Cố định `right`. Mọi chuỗi con có **đúng** `k` loại kết thúc tại `right` đều có **tối đa** `k` loại, nên đầu trái của nó `>= L(right)` — tức nó nằm trong cửa sổ `s[L(right)..right]`. Cửa sổ này chứa nó nên có **ít nhất** `k` loại, mà lại có tối đa `k` loại ⇒ cửa sổ có **đúng** `k` loại và dài hơn hoặc bằng. Ngược lại, nếu cửa sổ có ít hơn `k` loại thì mọi đoạn con của nó cũng vậy ⇒ không có đáp án nào kết thúc tại `right`.

**Vì sao Bài 6 cộng `right - left + 1`?** Với cửa sổ `s[left..right]` hợp lệ (tối đa `k` loại), các chuỗi con kết thúc tại `right` có tối đa `k` loại chính là `s[i..right]` với `i = left, left+1, ..., right` ⇒ đúng `right - left + 1` chuỗi. Rồi "đúng k" = "tối đa k" − "tối đa k−1".

In [39]:
# ---- Bài 5 ----
def longest_exactly_k_distinct(s: str, k: int) -> int:
    count, left, best = {}, 0, 0
    for right, ch in enumerate(s):
        count[ch] = count.get(ch, 0) + 1
        while len(count) > k:
            out = s[left]
            count[out] -= 1
            if count[out] == 0:
                del count[out]
            left += 1
        if len(count) == k:                      # chỉ đổi chỗ ghi nhận
            best = max(best, right - left + 1)
    return best if best > 0 else -1

assert longest_exactly_k_distinct("aabacbebebe", 3) == 7
assert longest_exactly_k_distinct("aaaa", 2) == -1
assert longest_exactly_k_distinct("aaabbb", 1) == 3
assert longest_exactly_k_distinct("eceba", 4) == 5

def exactly_bruteforce(s, k):
    best = -1
    for i in range(len(s)):
        for j in range(i, len(s)):
            if len(set(s[i:j + 1])) == k:
                best = max(best, j - i + 1)
    return best

# ---- Bài 6 ----
def count_at_most_k_distinct(s: str, k: int) -> int:
    if k < 0:
        return 0
    count, left, total = {}, 0, 0
    for right, ch in enumerate(s):
        count[ch] = count.get(ch, 0) + 1
        while len(count) > k:
            out = s[left]
            count[out] -= 1
            if count[out] == 0:
                del count[out]
            left += 1
        total += right - left + 1                # số chuỗi con hợp lệ kết thúc tại right
    return total

def count_exactly_k_distinct(s: str, k: int) -> int:
    return count_at_most_k_distinct(s, k) - count_at_most_k_distinct(s, k - 1)

assert count_exactly_k_distinct("abab", 2) == 6
assert count_exactly_k_distinct("aaa", 1) == 6
assert count_exactly_k_distinct("abc", 0) == 0
assert count_exactly_k_distinct("abc", 3) == 1

def count_exactly_bruteforce(s, k):
    return sum(1 for i in range(len(s)) for j in range(i, len(s)) if len(set(s[i:j + 1])) == k)

random.seed(56)
for _ in range(1000):
    s = "".join(random.choice("abcd") for _ in range(random.randint(1, 12)))
    k = random.randint(1, 5)
    assert longest_exactly_k_distinct(s, k) == exactly_bruteforce(s, k), (s, k)
    assert count_exactly_k_distinct(s, k) == count_exactly_bruteforce(s, k), (s, k)
print("Bài 5 và Bài 6 PASS")

Bài 5 và Bài 6 PASS


## 20. Template cuối cùng nên nhớ

```python
def longest_window(s, k):
    count, left, best = {}, 0, 0
    for right, ch in enumerate(s):
        count[ch] = count.get(ch, 0) + 1          # 1) THÊM phải
        while len(count) > k:                     # 2) VI PHẠM -> CO trái
            count[s[left]] -= 1
            if count[s[left]] == 0:
                del count[s[left]]                #    xóa key về 0!
            left += 1
        best = max(best, right - left + 1)        # 3) GHI NHẬN (sau khi co)
    return best
```

Câu thần chú: **"Thêm phải — `while` sai thì bỏ trái — ghi nhận."**

- Đổi điều kiện `len(count) > k` là sang được bài 3, 904, 1004...
- Đổi dòng ghi nhận thành `total += right - left + 1` là **đếm** được số đoạn (992).
- Cần stream / `O(1)` mỗi ký tự: lưu **vị trí cuối** trong `OrderedDict`, pop đầu hàng, `left = pos + 1`.
- Chỉ cần độ dài: có thể dùng mẹo **không co** (`if` thay `while`, đáp án `n - left`).

## 21. Checklist trước khi submit

- [ ] Tên method đúng: `lengthOfLongestSubstringKDistinct(self, s: str, k: int) -> int`.
- [ ] Xử lý **`k = 0`** ⇒ trả về `0` (khởi tạo `best = 0`, không phải `1`).
- [ ] `k` lớn hơn số loại ký tự ⇒ trả về `len(s)`.
- [ ] Thứ tự **thêm → co → ghi nhận**; ghi nhận **sau** vòng `while`.
- [ ] Khi đếm về 0 thì **`del`** key (đặc biệt nếu dùng `Counter`).
- [ ] Độ dài là `right - left + 1`.
- [ ] Nếu dùng vị trí cuối: `min(last, key=last.get)` (hoặc `OrderedDict` + `move_to_end`), và `left = last[victim] + 1`.
- [ ] Đã test ví dụ đề (`"eceba", 2 → 3`; `"aa", 1 → 2`), edge case (`k = 0`, 1 ký tự, `k` rất lớn, chữ hoa/thường) và đối chiếu brute force trên input ngẫu nhiên.
- [ ] Nói được độ phức tạp: Time `O(n)` (khấu hao: mỗi vị trí vào/ra cửa sổ tối đa 1 lần), Space `O(k)`.